# sketch2stl · ML2 — Revolve or mirror-extrude? (fine-tuned ResNet-18)

**Question it answers:** in *half + centreline* mode the person draws one half of an outline against the centreline. Should that half be **spun around the axis (revolve)** or **mirrored and extruded (mirror-extrude)**?

**Model category:** fine-tuned — torchvision **ResNet-18** pre-trained on ImageNet, all layers fine-tuned on rendered half-profiles.

| | |
|---|---|
| Revolve examples | 2,005 Fusion 360 Gallery designs that are solids of revolution (every extrude is a concentric circle/ring on one axis). The half is the exact cross-section of the final mesh, **checked automatically** (Pappus volume vs mesh volume, within 5 %). Plus procedurally generated turned parts (vase, ball, cone, knob, bowl, filleted shaft), because extrude-built parts are always straight steps. |
| Mirror examples | 4,906 extrude profiles that are mirror-symmetric (IoU ≥ 0.995), cut along their axis. |
| Ambiguous | a plain rectangle against the axis is a cylinder if revolved and a block if mirrored — the drawing cannot decide. Those **2,742** are flagged, left out of training and scoring, and reported separately. |
| Split | by CAD project (80/10/10), so variants of one part never sit on both sides |
| Baselines | majority class, and the app's current rule `suggest_kind` (curved outline → revolve) on the same halves |

No manual data collection for this part (agreed with the instructor) — every label comes from CAD geometry or a generator, checked by code.

Dataset: [pakiino/sketch2stl-datasets](https://huggingface.co/datasets/pakiino/sketch2stl-datasets) · Code: [SunKaiwen1023/sketch2stl](https://github.com/SunKaiwen1023/sketch2stl)

**Runtime → Change runtime type → T4 GPU**, then Run all (about 5 minutes on a GPU; it also runs on CPU, slower). The last section pushes the model only when `PUSH_TO_HUB = True`.


## 1 · Setup

In [ ]:
import os, sys, subprocess
REPO_URL   = "https://github.com/SunKaiwen1023/sketch2stl"
REPO_DIR   = "sketch2stl"
DATASET    = "pakiino/sketch2stl-datasets"
DATA_SUBDIR = "sketch2stl_datasets/ML2_revolve_vs_mirror"
MODEL_REPO = "pakiino/sketch2stl-revolve-or-mirror"
PUSH_TO_HUB = False
QUICK = bool(os.environ.get("SKETCH2STL_QUICK"))     # tiny run, for testing the notebook only

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shapely>=2.0", "scikit-image",
                "huggingface_hub", "onnx", "onnxruntime", "tabulate"], check=True)
sys.path.insert(0, os.path.abspath(REPO_DIR)); sys.path.insert(0, os.getcwd())
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

## 2 · The shared render code

`ml2_halves.py` is written next to the notebook. The **same** `render()` is used here and inside the app, so the model never sees a picture drawn differently from the one it was trained on.

Picture = 3 × 128 × 128: **ch0** the outline the person draws (edges lying on the axis are not drawn) · **ch1** the centreline · **ch2** the filled half.

In [ ]:
%%writefile ml2_halves.py
"""ML2 shared code: load the half-profiles, render them the way the app will, add hand noise.

The SAME render() is used for training and inside the app, so the model never sees
a picture drawn differently from the one it was trained on.

Picture (3 x 128 x 128, uint8), axis always vertical at the left:
    ch0  the outline the person draws (edges lying ON the axis are not drawn)
    ch1  the centreline
    ch2  the filled half (what close_to_axis / mirror_half will close it into)
"""
from __future__ import annotations

import csv
import numpy as np
from PIL import Image, ImageDraw

SIZE = 128
AXIS_X = 6          # pixel column of the centreline
PAD = 6


# ------------------------------------------------------------------ data
def load_halves(npz_path, csv_path):
    """Return (rows, shapes). shapes[i] = list of polygons, each [exterior, hole, hole, ...] (N,2) arrays,
    in the axis frame: x = distance from the axis (>= 0), y = along the axis."""
    z = np.load(npz_path)
    coords, ring_len = z["coords"], z["ring_len"]
    n_rings, n_polys, holes = z["n_rings"], z["n_polys"], z["holes_per_poly"]
    rows = list(csv.DictReader(open(csv_path)))
    starts = np.concatenate([[0], np.cumsum(ring_len)])
    shapes, r, h = [], 0, 0
    for i in range(len(rows)):
        rings = [coords[starts[r + k]:starts[r + k + 1]].astype(np.float64) for k in range(n_rings[i])]
        r += n_rings[i]
        polys, k = [], 0
        for _ in range(n_polys[i]):
            nh = int(holes[h]); h += 1
            polys.append(rings[k:k + 1 + nh]); k += 1 + nh
        shapes.append(polys)
    return rows, shapes


# ------------------------------------------------------------------ hand noise
def _densify(ring, step):
    out = [ring[0]]
    for a, b in zip(ring[:-1], ring[1:]):
        n = max(1, int(np.ceil(np.linalg.norm(b - a) / step)))
        out += [a + (b - a) * t for t in np.linspace(0, 1, n + 1)[1:]]
    return np.array(out)

def jitter(shape, rng, amount=0.012):
    """Low-frequency wobble along every ring, ~amount x part size. Points on the axis stay on it."""
    allp = np.vstack([r for poly in shape for r in poly])
    size = float(max(np.ptp(allp[:, 0]) + allp[:, 0].min(), np.ptp(allp[:, 1]), 1e-9))
    def area(r): return 0.5 * abs(float(np.dot(r[:-1, 0], r[1:, 1]) - np.dot(r[1:, 0], r[:-1, 1])))
    thick = np.sqrt(max(sum(area(poly[0]) for poly in shape), 1e-12))   # thin parts get gentler wobble
    amp = amount * min(size, 2.0 * thick)
    out = []
    for poly in shape:
        new = []
        for ring in poly:
            d = _densify(ring, size / 80)
            n = len(d)
            k = np.hanning(25); k /= k.sum()                  # smooth, low-frequency: a hand, not sandpaper
            noise = np.stack([np.convolve(rng.normal(0, 1, n + 24), k, "valid") for _ in range(2)], 1)
            noise = noise[:n] / max(noise.std(), 1e-9) * amp
            on_axis = d[:, 0] < 1e-6 * size
            noise[on_axis, 0] = 0.0
            dd = d + noise
            dd[:, 0] = np.maximum(dd[:, 0], 0.0)
            dd[-1] = dd[0]
            new.append(dd)
        out.append(new)
    # mild global stretch: people do not draw to scale
    sx, sy = np.exp(rng.normal(0, 0.08, 2))
    return [[r * [sx, sy] for r in poly] for poly in out]


# ------------------------------------------------------------------ rendering
def _frame(shape):
    allp = np.vstack([r for poly in shape for r in poly])
    w = max(float(allp[:, 0].max()), 1e-9)
    y0, y1 = float(allp[:, 1].min()), float(allp[:, 1].max())
    h = max(y1 - y0, 1e-9)
    s = min((SIZE - AXIS_X - PAD) / w, (SIZE - 2 * PAD) / h)
    cy = (y0 + y1) / 2
    return lambda p: np.stack([AXIS_X + p[:, 0] * s, SIZE / 2 - (p[:, 1] - cy) * s], 1)

def render(shape, line_w=2):
    """shape = list of polygons [[exterior, holes...], ...] in the axis frame. Returns uint8 (3, SIZE, SIZE)."""
    f = _frame(shape)
    allx = np.vstack([r for poly in shape for r in poly])[:, 0]
    tol = 1e-3 * max(float(allx.max()), 1e-9)
    c0, c1, c2 = (Image.new("L", (SIZE, SIZE), 0) for _ in range(3))
    d0, d1, d2 = ImageDraw.Draw(c0), ImageDraw.Draw(c1), ImageDraw.Draw(c2)
    d1.line([(AXIS_X, 0), (AXIS_X, SIZE - 1)], fill=255, width=1)
    for poly in shape:
        ext = f(poly[0]); d2.polygon([tuple(p) for p in ext], fill=255)
        for hole in poly[1:]:
            d2.polygon([tuple(p) for p in f(hole)], fill=0)
        for ring in poly:
            q = f(ring)
            for (a, b), (ra, rb) in zip(zip(q[:-1], q[1:]), zip(ring[:-1], ring[1:])):
                if ra[0] < tol and rb[0] < tol:      # this edge lies on the axis: nobody draws it
                    continue
                d0.line([tuple(a), tuple(b)], fill=255, width=line_w)
    return np.stack([np.asarray(c) for c in (c0, c1, c2)])


# ------------------------------------------------------------------ app side
def shape_from_drawing(half_points, axis_point, axis_dir):
    """The app's half (open polyline, mm) + its Axis -> the same axis-frame shape render() expects."""
    p = np.asarray(half_points, float).reshape(-1, 2)
    d = np.asarray(axis_dir, float); d = d / np.linalg.norm(d)
    rel = p - np.asarray(axis_point, float)
    x = np.abs(rel[:, 0] * d[1] - rel[:, 1] * d[0])
    y = rel @ d
    ring = np.stack([x, y], 1)
    # close it the way close_to_axis does: drop to the axis at both ends
    ring = np.vstack([[0.0, ring[0, 1]], ring, [0.0, ring[-1, 1]], [0.0, ring[0, 1]]])
    return [[ring]]


# ------------------------------------------------------------------ procedural turned parts
# Every revolvable part in the Fusion set was BUILT FROM EXTRUDES, so its half is all
# straight steps. Nobody draws a vase that way. Without these, the model would learn
# "curved outline -> mirror", the opposite of what a person drawing half a ball means.
# Synthetic, labelled as such (source = procedural) and scored separately.
TURNED = ["vase", "ball", "cone", "knob", "bowl", "filleted_steps"]

def _smooth_profile(rng, n_ctrl):
    ys = np.linspace(0, 1, n_ctrl); rs = rng.uniform(0.15, 1.0, n_ctrl)
    y = np.linspace(0, 1, 80)
    r = np.interp(y, ys, rs)
    k = np.hanning(15); k /= k.sum()
    r = np.convolve(np.pad(r, 7, mode="edge"), k, "valid")
    return y, np.maximum(r, 0.08)

def turned_part(rng, kind=None):
    """One random half-profile of a turned part, axis frame (x = radius, y = height)."""
    kind = kind or TURNED[rng.integers(len(TURNED))]
    if kind == "vase":
        y, r = _smooth_profile(rng, rng.integers(4, 8))
        pts = np.vstack([[0, 0], np.stack([r, y], 1), [0, 1]])
    elif kind == "ball":
        a = np.linspace(-np.pi / 2, np.pi / 2, 60)
        pts = np.stack([np.cos(a), np.sin(a)], 1) * 0.5 + [0, 0.5]
        cut = rng.uniform(0, 0.3)                               # sometimes a flat base
        pts = pts[pts[:, 1] >= cut]; pts = np.vstack([[0, pts[0, 1]], pts])
    elif kind == "cone":
        top = rng.uniform(0, 0.5)
        pts = np.array([[0, 0], [1, 0], [top, 1], [0, 1]])
    elif kind == "knob":
        stem, cap_h = rng.uniform(0.2, 0.5), rng.uniform(0.3, 0.6)
        a = np.linspace(0, np.pi / 2, 30)
        cap = np.stack([np.cos(a), (1 - cap_h) + cap_h * np.sin(a)], 1)
        pts = np.vstack([[0, 0], [stem, 0], [stem, 1 - cap_h], cap, [0, 1]])
    elif kind == "bowl":
        a = np.linspace(0, np.pi / 2, 40); t = rng.uniform(0.08, 0.2)
        outer = np.stack([np.sin(a), 1 - np.cos(a)], 1)
        inner = np.stack([np.sin(a) * (1 - t), 1 - np.cos(a) * (1 - t) + t * 0.0], 1)[::-1]
        pts = np.vstack([[0, 0], outer, inner[:-1], [0, inner[-1, 1]]])
    else:  # filleted_steps: a stepped shaft with rounded corners
        ys = np.sort(rng.uniform(0, 1, rng.integers(2, 4))); rs = rng.uniform(0.2, 1.0, len(ys) + 1)
        pts = [[0, 0]]; y0 = 0.0
        for yy, rr in zip(list(ys) + [1.0], rs):
            pts += [[rr, y0], [rr, yy]]; y0 = yy
        pts += [[0, 1]]
        from shapely.geometry import Polygon
        g = Polygon(pts).buffer(-0.04, join_style=1).buffer(0.04, join_style=1)
        if g.is_empty or g.geom_type != "Polygon":
            return turned_part(rng, "vase")
        p = np.asarray(g.exterior.coords); p[:, 0] = np.where(p[:, 0] < 0.05, 0.0, p[:, 0])
        pts = p
    pts = np.asarray(pts, float)
    pts[:, 0] *= rng.uniform(0.3, 2.5)                           # aspect: thin shaft .. wide disc
    if not np.allclose(pts[0], pts[-1]): pts = np.vstack([pts, pts[:1]])
    return [[pts]], kind


## 3 · Download the half-profiles

Made by `build_halves.py` from the Fusion 360 Gallery (code below for reference, it needs the raw dataset and is not run here).

In [ ]:
from huggingface_hub import snapshot_download
LOCAL = os.environ.get("SKETCH2STL_ML2_DATA")              # only for offline testing
if LOCAL:
    D = LOCAL
else:
    root = snapshot_download(DATASET, repo_type="dataset", allow_patterns=[DATA_SUBDIR + "/*"])
    D = os.path.join(root, DATA_SUBDIR)
print(sorted(os.listdir(D)))
print(open(os.path.join(D, "summary.json")).read())

<details><summary>build_halves.py (how the data was made)</summary>

```python
#!/usr/bin/env python3
"""Fusion 360 Gallery -> "half + centreline" profiles for ML2 (revolve vs mirror-extrude).

    python build_halves.py --src r1.0.1 --out halves --start 0 --stop 2000   # in chunks
    python build_halves.py --out halves --merge                                # then merge

WHAT IT MAKES
-------------
Each sample is what a user would draw in half mode: a 2-D outline on ONE side of a
vertical centreline (x = 0, the axis, y along it, x >= 0), in sketch cm.

  revolve  Designs whose every extrude is a concentric circle / ring on one shared
           axis (an extruded circle == a revolved rectangle). The half is the exact
           cross-section of the FINAL mesh, sliced by a plane through the axis.
           Checked automatically: Pappus volume of the half vs the mesh volume.
  mirror   Extrude profiles (not all-circle) that are mirror-symmetric about some
           axis (IoU >= 0.995). The half is the profile cut along that axis.
           One sample per design, so near-duplicates do not pile up.

  ambiguous = the half is a plain rectangle touching the axis. Revolved it is a
           cylinder, mirrored it is a block: the drawing cannot decide. Kept in the
           files, flagged, and left out of training/scoring by the notebook.

Split by PROJECT (metadata.parent_project) 80/10/10 with a fixed hash, so variants
of one part never sit on both sides.

LICENCE: derived from the Fusion 360 Gallery Dataset (non-commercial research use).
Needs only numpy + shapely.
"""
from __future__ import annotations

import argparse, csv, glob, hashlib, json, math, os, sys
from collections import Counter

import numpy as np
from shapely.geometry import LineString, MultiPolygon, Point, Polygon, box
from shapely.ops import polygonize, unary_union
from shapely import affinity


# ------------------------------------------------------------------ small helpers
def v3(p): return np.array([p["x"], p["y"], p["z"]], float)
def P(p): return np.array([p["x"], p["y"]], float)

def split_of(project: str) -> str:
    h = int(hashlib.md5(project.encode()).hexdigest()[:8], 16) % 100
    return "train" if h < 80 else ("val" if h < 90 else "test")

def polys(g):
    if g.is_empty: return []
    if isinstance(g, Polygon): return [g]
    if isinstance(g, MultiPolygon): return list(g.geoms)
    return [x for x in getattr(g, "geoms", []) if isinstance(x, Polygon)]

def read_obj(path):
    V, F = [], []
    with open(path) as fh:
        for line in fh:
            if line.startswith("v "): V.append(line.split()[1:4])
            elif line.startswith("f "): F.append([x.split("/")[0] for x in line.split()[1:4]])
    return np.array(V, float), np.array(F, int) - 1


# ------------------------------------------------------------------ sketch curve sampling
def bspline(c, n=48):
    cp = np.array([[q["x"], q["y"]] for q in c["control_points"]]); k = c["degree"]
    t = np.array(c["knots"], float); w = np.array(c.get("weights") or [1.0] * len(cp), float)
    out = []
    for u in np.linspace(t[k], t[-k - 1], n):
        N = np.array([1.0 if (t[i] <= u < t[i + 1]) or (u == t[-k - 1] and t[i] < u <= t[i + 1]) else 0.0
                      for i in range(len(t) - 1)])
        for d in range(1, k + 1):
            N2 = np.zeros(len(t) - 1 - d)
            for i in range(len(N2)):
                a = (u - t[i]) / (t[i + d] - t[i]) * N[i] if t[i + d] != t[i] else 0
                b = (t[i + d + 1] - u) / (t[i + d + 1] - t[i + 1]) * N[i + 1] if t[i + d + 1] != t[i + 1] else 0
                N2[i] = a + b
            N = N2
        N = N[:len(cp)] * w; s = N.sum()
        out.append((N[:, None] * cp).sum(0) / s if s > 0 else cp[-1])
    return np.array(out)

def sample(c):
    t = c["type"]
    if t == "Line3D": return np.array([P(c["start_point"]), P(c["end_point"])])
    if t == "Circle3D":
        a = np.linspace(0, 2 * math.pi, 97); return P(c["center_point"]) + c["radius"] * np.stack([np.cos(a), np.sin(a)], 1)
    if t == "Arc3D":
        rv = c["reference_vector"]; a0 = math.atan2(rv["y"], rv["x"]); sg = 1 if c["normal"]["z"] >= 0 else -1
        a = a0 + sg * np.linspace(c["start_angle"], c["end_angle"], max(8, int(abs(c["end_angle"] - c["start_angle"]) / 0.1)))
        return P(c["center_point"]) + c["radius"] * np.stack([np.cos(a), np.sin(a)], 1)
    if t == "Ellipse3D":
        m = c["major_axis"]; g = math.atan2(m["y"], m["x"]); a = np.linspace(0, 2 * math.pi, 97)
        x = c["major_axis_radius"] * np.cos(a); y = c["minor_axis_radius"] * np.sin(a)
        return P(c["center_point"]) + np.stack([x * math.cos(g) - y * math.sin(g), x * math.sin(g) + y * math.cos(g)], 1)
    if t == "NurbsCurve3D": return bspline(c)
    raise ValueError(t)

def chain(curves):
    segs = [sample(c) for c in curves]
    if len(segs) == 1: return segs[0]
    pts = list(segs[0])
    if min(np.linalg.norm(segs[0][0] - segs[1][0]), np.linalg.norm(segs[0][0] - segs[1][-1])) < 1e-6: pts = pts[::-1]
    for s in segs[1:]:
        if np.linalg.norm(s[-1] - pts[-1]) < np.linalg.norm(s[0] - pts[-1]): s = s[::-1]
        pts.extend(s[1:])
    return np.array(pts)


# ------------------------------------------------------------------ revolve detection (same rule as axisym.py)
def loop_circle(loop):
    cs = loop["profile_curves"]
    if not cs or any(c["type"] not in ("Circle3D", "Arc3D") for c in cs): return None
    c0 = v3(cs[0]["center_point"]); r0 = cs[0]["radius"]
    for c in cs:
        if np.linalg.norm(v3(c["center_point"]) - c0) > 1e-4 * max(1, r0) or abs(c["radius"] - r0) > 1e-4 * max(1, r0): return None
    return c0, r0

def revolve_axis(d):
    """(point, unit direction, max radius) if the whole design is axisymmetric, else None."""
    E = d["entities"]
    feats = [E[t["entity"]] for t in d["timeline"] if E[t["entity"]]["type"] == "ExtrudeFeature"]
    if not feats: return None
    axes, rmax = [], 0.0
    for fe in feats:
        pts = []
        for pr in fe["profiles"]:
            sk = E[pr["sketch"]]; T = sk["transform"]
            o, xa, ya, za = v3(T["origin"]), v3(T["x_axis"]), v3(T["y_axis"]), v3(T["z_axis"])
            for lp in sk["profiles"][pr["profile"]]["loops"]:
                c = loop_circle(lp)
                if c is None: return None
                pts.append(o + c[0][0] * xa + c[0][1] * ya); rmax = max(rmax, c[1])
        Pm = np.array(pts)
        if not len(Pm): return None
        off = Pm - Pm[0] - np.outer((Pm - Pm[0]) @ za, za)
        if np.max(np.linalg.norm(off, axis=1)) > 1e-3: return None
        axes.append((Pm[0], za))
    p0, a0 = axes[0]
    for p, a in axes[1:]:
        if abs(abs(a @ a0) - 1) > 1e-6: return None
        w = p - p0
        if np.linalg.norm(w - (w @ a0) * a0) > 1e-3: return None
    return p0, a0 / np.linalg.norm(a0), rmax


# ------------------------------------------------------------------ revolve half = mesh section
def _inside(points, V, F, direction=np.array([0.5773, 0.5791, 0.5755])):
    """Ray-parity point-in-mesh test (Moller-Trumbore), vectorised over triangles."""
    d = direction / np.linalg.norm(direction)
    v0, v1, v2 = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    e1, e2 = v1 - v0, v2 - v0
    h = np.cross(d, e2); det = np.einsum("ij,ij->i", e1, h)
    ok = np.abs(det) > 1e-14; inv = np.where(ok, 1.0 / np.where(ok, det, 1.0), 0.0)
    out = []
    for p in points:
        sv = p - v0; uu = np.einsum("ij,ij->i", sv, h) * inv
        q = np.cross(sv, e1); vv = (q @ d) * inv; tt = np.einsum("ij,ij->i", e2, q) * inv
        hit = ok & (uu >= 0) & (vv >= 0) & (uu + vv <= 1) & (tt > 1e-12)
        out.append(int(hit.sum()) % 2 == 1)
    return np.array(out)

def section_half(V, F, p0, a, max_levels=60):
    """Exact r-z half-section of an axisymmetric part made of extrudes.

    Every boundary of such a part is r = const or z = const, so the section is a union
    of grid cells between the distinct radii and the distinct heights found in the mesh.
    Each cell is kept if its centre, placed in 3-D, is inside the mesh (ray parity).
    x = radius, y = height along the axis.
    """
    D = V - p0
    z = D @ a
    r = np.linalg.norm(D - np.outer(z, a), axis=1)
    size = float(max(np.ptp(z), r.max(), 1e-9))
    q = 1e-4 * size
    R = np.unique(np.round(np.concatenate([[0.0], r]) / q)) * q
    Z = np.unique(np.round(z / q)) * q
    if len(R) > max_levels or len(Z) > max_levels: return None      # tapered / not built from rings
    u = np.cross(a, [1.0, 0, 0]) if abs(a[0]) < 0.9 else np.cross(a, [0, 1.0, 0]); u /= np.linalg.norm(u)
    cells, centres = [], []
    for i in range(len(R) - 1):
        for j in range(len(Z) - 1):
            rc, zc = (R[i] + R[i + 1]) / 2, (Z[j] + Z[j + 1]) / 2
            cells.append((R[i], Z[j], R[i + 1], Z[j + 1])); centres.append(p0 + rc * u + zc * a)
    if not cells: return None
    inside = _inside(np.array(centres), V, F)
    keep = [box(*c) for c, k in zip(cells, inside) if k]
    if not keep: return None
    return unary_union(keep).buffer(1e-7 * size, join_style=2).buffer(-1e-7 * size, join_style=2).simplify(1e-6 * size)

def mesh_volume(V, F):
    a, b, c = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    return abs(float(np.einsum("ij,ij->i", a, np.cross(b, c)).sum()) / 6.0)

def pappus(half):
    return float(sum(2 * math.pi * p.area * p.centroid.x for p in polys(half)))


# ------------------------------------------------------------------ mirror half = symmetric extrude profile
def iou(p, q):
    u = p.union(q).area
    return p.intersection(q).area / u if u > 0 else 0.0

def reflect(g, o, ang):
    c2, s2 = math.cos(2 * ang), math.sin(2 * ang); cx, cy = o
    return affinity.affine_transform(g, [c2, s2, s2, -c2, cx - (c2 * cx + s2 * cy), cy - (s2 * cx - c2 * cy)])

def best_mirror(poly):
    """Try the few axes a CAD designer would use; return (iou, origin, angle)."""
    c = poly.centroid; minx, miny, maxx, maxy = poly.bounds
    origins = [(c.x, c.y), ((minx + maxx) / 2, (miny + maxy) / 2)]
    pts = np.asarray(poly.exterior.coords); pts = pts - pts.mean(0)
    w, vec = np.linalg.eigh(pts.T @ pts)
    angs = {0.0, math.pi / 2, math.atan2(vec[1, 0], vec[0, 0]) % math.pi, math.atan2(vec[1, 1], vec[0, 1]) % math.pi}
    best = (0.0, None, None)
    for o in origins:
        for a in angs:
            s = iou(poly, reflect(poly, o, a))
            if s > best[0]: best = (s, o, a)
    return best

def to_half(poly, o, ang):
    """Rotate so the mirror axis is the y axis (x = 0), keep x >= 0."""
    g = affinity.translate(poly, -o[0], -o[1])
    g = affinity.rotate(g, 90.0 - math.degrees(ang), origin=(0, 0))
    minx, miny, maxx, maxy = g.bounds; big = 2 * max(maxx - minx, maxy - miny, 1e-6)
    return g.intersection(box(0, miny - big, maxx + big, maxy + big))

def step_profile(d, fe):
    E = d["entities"]; shapes = []; all_circles = True
    sks = {p["sketch"] for p in fe["profiles"]}
    if len(sks) != 1: return None, True
    sk = E[next(iter(sks))]
    for pr in fe["profiles"]:
        loops = sk["profiles"][pr["profile"]]["loops"]; outer, holes = None, []
        for lp in loops:
            if loop_circle(lp) is None: all_circles = False
            ring = chain(lp["profile_curves"])
            if len(ring) < 3: continue
            if lp["is_outer"]: outer = ring
            else: holes.append(ring)
        if outer is None: continue
        g = Polygon(outer, [h for h in holes if len(h) >= 3])
        shapes.append(g if g.is_valid else g.buffer(0))
    if not shapes: return None, all_circles
    return unary_union(shapes), all_circles


# ------------------------------------------------------------------ ambiguity
def is_axis_rectangle(half, tol=0.02):
    """A plain rectangle with one side on the axis: cylinder if revolved, block if mirrored."""
    ps = polys(half)
    if len(ps) != 1 or len(ps[0].interiors): return False
    p = ps[0]; minx, miny, maxx, maxy = p.bounds
    return minx < tol * max(maxx, 1e-9) and p.area > (1 - tol) * (maxx - minx) * (maxy - miny)


# ------------------------------------------------------------------ main
FIELDS = ["sample_id", "design_id", "project", "split", "kind", "ambiguous", "n_steps", "source",
          "width_cm", "height_cm", "n_rings", "check"]

def extract(a):
    rec = os.path.join(a.src, "reconstruction")
    files = sorted(glob.glob(os.path.join(rec, "*.json")))[a.start:a.stop]
    rows, rings_all, why = [], [], Counter()
    for f in files:
        did = os.path.basename(f)[:-5]
        try: d = json.load(open(f))
        except Exception: why["bad_json"] += 1; continue
        proj = d["metadata"].get("parent_project", did.rsplit("_", 1)[0])
        E = d["entities"]
        feats = [E[t["entity"]] for t in d["timeline"] if E[t["entity"]]["type"] == "ExtrudeFeature"]
        ax = revolve_axis(d)
        half, kind, check, source = None, None, "", ""
        try:
            if ax is not None:
                if sum(fe["operation"] in ("NewBodyFeatureOperation", "NewComponentFeatureOperation") for fe in feats) > 1:
                    why["revolve_multi_body"] += 1; continue          # overlapping bodies break parity + volume
                objp = os.path.join(rec, did + ".obj")
                if not os.path.exists(objp): why["revolve_no_obj"] += 1; continue
                V, F = read_obj(objp)
                half = section_half(V, F, ax[0], ax[1])
                if half is None or half.is_empty: why["revolve_slice_failed"] += 1; continue
                vm, vp = mesh_volume(V, F), pappus(half)
                err = abs(vp - vm) / max(vm, 1e-12)
                if err > 0.05: why["revolve_volume_mismatch"] += 1; continue
                kind, check, source = "revolve", f"pappus_err={err:.4f}", "final_mesh"
            else:
                for k, fe in enumerate(feats):
                    prof, circ = step_profile(d, fe)
                    if prof is None or circ or prof.area <= 0: continue
                    s, o, ang = best_mirror(polys(prof)[0] if isinstance(prof, MultiPolygon) else prof)
                    if s < 0.995: continue
                    half = to_half(prof, o, ang)
                    if half.is_empty or half.area < 0.3 * prof.area: half = None; continue
                    kind, check, source = "mirror", f"iou={s:.4f}", f"step{k}"
                    break
                if half is None: why["mirror_none_symmetric"] += 1; continue
        except Exception as e:
            why["error:" + type(e).__name__] += 1; continue
        ps = polys(half)
        if not ps: why["empty"] += 1; continue
        rings = []
        for p in ps:
            rings.append(np.asarray(p.exterior.coords, np.float32))
            rings += [np.asarray(r.coords, np.float32) for r in p.interiors]
        minx, miny, maxx, maxy = unary_union(ps).bounds
        sid = f"{did}_{kind}"
        rows.append(dict(sample_id=sid, design_id=did, project=proj, split=split_of(proj), kind=kind,
                         ambiguous=int(is_axis_rectangle(half)), n_steps=len(feats), source=source,
                         width_cm=round(maxx - minx, 5), height_cm=round(maxy - miny, 5),
                         n_rings=len(rings), check=check))
        rings_all.append((len(ps), [len(p.interiors) for p in ps], rings))
        why[kind] += 1
    os.makedirs(a.out, exist_ok=True)
    tag = f"{a.start:05d}"
    coords = np.concatenate([r for _, _, rs in rings_all for r in rs]) if rings_all else np.zeros((0, 2), np.float32)
    ring_len = np.array([len(r) for _, _, rs in rings_all for r in rs], np.int32)
    n_rings = np.array([len(rs) for _, _, rs in rings_all], np.int32)
    holes = np.array([h for _, hs, _ in rings_all for h in hs], np.int32)       # holes per polygon
    n_polys = np.array([n for n, _, _ in rings_all], np.int32)
    np.savez_compressed(os.path.join(a.out, f"part_{tag}.npz"), coords=coords, ring_len=ring_len,
                        n_rings=n_rings, n_polys=n_polys, holes_per_poly=holes)
    with open(os.path.join(a.out, f"part_{tag}.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS); w.writeheader(); w.writerows(rows)
    print(dict(why), flush=True)

def merge(a):
    parts = sorted(glob.glob(os.path.join(a.out, "part_*.npz")))
    C, RL, NR, NP, HP, rows = [], [], [], [], [], []
    for p in parts:
        z = np.load(p); C.append(z["coords"]); RL.append(z["ring_len"]); NR.append(z["n_rings"])
        NP.append(z["n_polys"]); HP.append(z["holes_per_poly"])
        rows += list(csv.DictReader(open(p[:-4] + ".csv")))
    np.savez_compressed(os.path.join(a.out, "halves.npz"), coords=np.concatenate(C), ring_len=np.concatenate(RL),
                        n_rings=np.concatenate(NR), n_polys=np.concatenate(NP), holes_per_poly=np.concatenate(HP))
    with open(os.path.join(a.out, "halves.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS); w.writeheader(); w.writerows(rows)
    summ = {"samples": len(rows),
            "by_kind": Counter(r["kind"] for r in rows),
            "ambiguous_by_kind": Counter(r["kind"] for r in rows if r["ambiguous"] == "1"),
            "by_split_kind": Counter(f'{r["split"]}/{r["kind"]}' for r in rows),
            "projects": len({r["project"] for r in rows})}
    json.dump(summ, open(os.path.join(a.out, "summary.json"), "w"), indent=2)
    print(json.dumps(summ, indent=2))

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", default="r1.0.1"); ap.add_argument("--out", default="halves")
    ap.add_argument("--start", type=int, default=0); ap.add_argument("--stop", type=int, default=None)
    ap.add_argument("--merge", action="store_true")
    a = ap.parse_args()
    merge(a) if a.merge else extract(a)

```
</details>

## 4 · Build the train / val / test sets

* CAD halves, **ambiguous ones excluded** (kept aside for section 9)
* plus procedural turned parts for the revolve class, split the same way
* training pictures get fresh hand-wobble every epoch; val/test get one fixed wobble each, so the test is not a set of perfect CAD lines

In [ ]:
import numpy as np, csv
from collections import Counter
import ml2_halves as H

rows, shapes = H.load_halves(os.path.join(D, "halves.npz"), os.path.join(D, "halves.csv"))
LABEL = {"mirror": 0, "revolve": 1}; NAMES = ["mirror", "revolve"]

items = {"train": [], "val": [], "test": []}; ambiguous = []
for r, s in zip(rows, shapes):
    group = r["kind"] + (" (cad, stepped)" if r["kind"] == "revolve" and int(r["n_steps"]) >= 2 else
                         " (cad, single step)" if r["kind"] == "revolve" else " (cad)")
    it = dict(shape=s, y=LABEL[r["kind"]], group=group, id=r["sample_id"])
    (ambiguous if r["ambiguous"] == "1" else items[r["split"]]).append(it)

N_PROC = {"train": 1500, "val": 200, "test": 200}
for split, n in N_PROC.items():
    rng = np.random.default_rng({"train": 1, "val": 2, "test": 3}[split])
    for k in range(n if not QUICK else n // 10):
        s, kind = H.turned_part(rng)
        items[split].append(dict(shape=s, y=1, group=f"revolve (procedural)", id=f"proc_{split}_{k}_{kind}"))

if QUICK:
    rng = np.random.default_rng(0)
    for k in items: items[k] = [items[k][i] for i in rng.permutation(len(items[k]))[:300]]
for k, v in items.items():
    print(f"{k:5s} {len(v):5d}  ", dict(Counter(i['group'] for i in v)))
print("ambiguous (held aside):", len(ambiguous))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class Halves(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
        if not train:                                   # one fixed wobble per test picture
            self.cache = [H.render(H.jitter(it["shape"], np.random.default_rng(10_000 + i)))
                          for i, it in enumerate(items)]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        img = H.render(H.jitter(it["shape"], np.random.default_rng())) if self.train else self.cache[i]
        x = (torch.from_numpy(img).float() / 255.0 - MEAN) / STD
        return x, it["y"]

ds = {k: Halves(v, k == "train") for k, v in items.items()}
# balance the classes: weight each picture by 1 / class frequency
ytr = np.array([it["y"] for it in items["train"]])
w = 1.0 / np.bincount(ytr)[ytr]
sampler = torch.utils.data.WeightedRandomSampler(torch.tensor(w, dtype=torch.double), len(w), replacement=True)
NW = 2 if DEVICE == "cuda" else 0
dl = {"train": DataLoader(ds["train"], batch_size=64, sampler=sampler, num_workers=NW),
      "val": DataLoader(ds["val"], batch_size=128, num_workers=NW),
      "test": DataLoader(ds["test"], batch_size=128, num_workers=NW)}
print({k: len(v) for k, v in ds.items()})

### What the model sees

In [ ]:
import matplotlib.pyplot as plt
def show(img):
    rgb = np.full((128, 128, 3), 255, np.uint8)
    rgb[img[2] > 0] = (200, 220, 255); rgb[img[1] > 0] = (230, 120, 0); rgb[img[0] > 0] = (20, 20, 20)
    return rgb
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
pick = np.random.default_rng(4)
groups = ["revolve (cad, stepped)", "revolve (procedural)", "mirror (cad)"]
for row, g in zip(axes, groups):
    pool = [it for it in items["train"] if it["group"] == g]
    for ax in row:
        it = pool[pick.integers(len(pool))]
        ax.imshow(show(H.render(H.jitter(it["shape"], pick)))); ax.set_title(g, fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

## 5 · Fine-tune ResNet-18

In [ ]:
import torch.nn as nn, time, torchvision
WEIGHTS = None if os.environ.get("SKETCH2STL_NO_PRETRAINED") else torchvision.models.ResNet18_Weights.IMAGENET1K_V1
model = torchvision.models.resnet18(weights=WEIGHTS)       # ImageNet weights = the "fine-tuned" part
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(DEVICE)

EPOCHS = 1 if QUICK else 8
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(dl["train"]))
lossf = nn.CrossEntropyLoss()

def predict(loader):
    model.eval(); L, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            L.append(model(x.to(DEVICE)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

hist = []
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    for x, y in dl["train"]:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = lossf(model(x), y); loss.backward(); opt.step(); sched.step()
        tot += loss.item() * len(y)
    lv, yv = predict(dl["val"])
    acc = (lv.argmax(1) == yv).float().mean().item()
    hist.append((tot / len(ds["train"]), acc))
    print(f"epoch {ep + 1}/{EPOCHS}  loss {hist[-1][0]:.4f}  val acc {acc:.4f}  ({time.time() - t0:.0f}s)")

## 6 · Calibrate the confidence

The app shows this number to the person (*"Looks like a revolve — 87 %"*), so it has to mean something. One temperature `T` is fitted on the **validation** set; logits are divided by `T`.

In [ ]:
def ece(probs, y, bins=10):
    conf, pred = probs.max(1); e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.float().mean() * abs((pred[m] == y[m]).float().mean() - conf[m].mean())
    return float(e)

lv, yv = predict(dl["val"])
logT = torch.zeros(1, requires_grad=True)
o = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
def closure():
    o.zero_grad(); l = nn.functional.cross_entropy(lv / logT.exp(), yv); l.backward(); return l
o.step(closure)
T = float(logT.exp().item())
print(f"temperature T = {T:.3f}")
print(f"val ECE before {ece(lv.softmax(1), yv):.4f}  after {ece((lv / T).softmax(1), yv):.4f}")

## 7 · Test: model vs baselines

* **majority** — always say the most common class
* **rule** — the app's current `suggest_kind` (repo `sketch2stl/suggest.py`): *curved outline with few corners → revolve*. Run on the same wobbly halves, in millimetres, with the same `Axis` type the app uses.
* **model** — this ResNet-18

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sketch2stl.suggest import suggest_kind
from sketch2stl.types import Axis

lt, yt = predict(dl["test"]); pt = (lt / T).softmax(1); y_model = pt.argmax(1).numpy(); y_true = yt.numpy()

def drawn_points(shape, rng):
    """What the person would have drawn: the largest outline, wobbled, minus the part on the axis, in mm."""
    poly = max(H.jitter(shape, rng), key=lambda p: len(p[0]))
    ring = poly[0]; tol = 1e-3 * max(ring[:, 0].max(), 1e-9)
    pts = ring[ring[:, 0] > tol]
    if len(pts) < 4: pts = ring
    span = max(np.ptp(ring[:, 1]), ring[:, 0].max(), 1e-9)
    return pts / span * 60.0                     # a ~60 mm drawing, like on the canvas

AX = Axis(point=np.array([0.0, 0.0]), direction=np.array([0.0, 1.0]))
# suggest_kind answers "revolve" or "extrude" (= mirror-extrude in half mode)
y_rule = np.array([1 if suggest_kind(drawn_points(it["shape"], np.random.default_rng(10_000 + i)), AX).value == "revolve" else 0
                   for i, it in enumerate(items["test"])])
y_major = np.full_like(y_true, np.bincount(np.array([it["y"] for it in items["train"]])).argmax())

def row(name, yp):
    return dict(arm=name, accuracy=round(accuracy_score(y_true, yp), 4),
                macro_f1=round(f1_score(y_true, yp, average="macro"), 4),
                revolve_recall=round(((yp == 1) & (y_true == 1)).sum() / max((y_true == 1).sum(), 1), 4),
                mirror_recall=round(((yp == 0) & (y_true == 0)).sum() / max((y_true == 0).sum(), 1), 4))
import pandas as pd
results = pd.DataFrame([row("majority", y_major), row("rule (suggest_kind)", y_rule), row("ResNet-18 (fine-tuned)", y_model)])
display(results)

groups = np.array([it["group"] for it in items["test"]])
by = []
for g in sorted(set(groups)):
    m = groups == g
    by.append(dict(group=g, n=int(m.sum()), rule=round((y_rule[m] == y_true[m]).mean(), 3),
                   model=round((y_model[m] == y_true[m]).mean(), 3)))
by = pd.DataFrame(by); display(by)
print("model test ECE:", round(ece(pt, yt), 4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, yp) in zip(axes[:2], [("rule (suggest_kind)", y_rule), ("ResNet-18", y_model)]):
    cm = confusion_matrix(y_true, yp, labels=[0, 1]); ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(NAMES); ax.set_yticks([0, 1]); ax.set_yticklabels(NAMES)
    for i in range(2):
        for j in range(2): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
ax = axes[2]
x = np.arange(len(by)); ax.bar(x - 0.2, by["rule"], 0.4, label="rule"); ax.bar(x + 0.2, by["model"], 0.4, label="ResNet-18")
ax.set_xticks(x); ax.set_xticklabels(by["group"], rotation=20, ha="right", fontsize=8); ax.set_ylim(0, 1); ax.legend(); ax.set_title("accuracy by group")
plt.tight_layout(); os.makedirs("ml2_out", exist_ok=True); plt.savefig("ml2_out/results.png", dpi=140); plt.show()

### Where it goes wrong

In [ ]:
wrong = np.where(y_model != y_true)[0]
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, i in zip(axes.ravel(), wrong[:16]):
    ax.imshow(show(ds["test"].cache[i])); ax.axis("off")
    ax.set_title(f"true {NAMES[y_true[i]]}\nsaid {NAMES[y_model[i]]} {pt[i].max():.0%}", fontsize=7)
for ax in axes.ravel()[len(wrong[:16]):]: ax.axis("off")
plt.tight_layout(); plt.show(); print(len(wrong), "of", len(y_true), "wrong")

## 8 · The ambiguous halves

A rectangle against the axis is a cylinder or a block — both are right. The model never saw these. A well-behaved model should **not** be confident here; the app then shows both 3-D previews and lets the person pick.

In [ ]:
amb = ambiguous[:400] if QUICK else ambiguous
la, _ = predict(DataLoader(Halves(amb, False), batch_size=128))
pa = (la / T).softmax(1)
conf = pa.max(1).values.numpy()
print(f"{len(amb)} ambiguous halves: median confidence {np.median(conf):.2f}, "
      f"{(conf < 0.7).mean():.0%} below 70 %, says revolve for {(pa.argmax(1) == 1).float().mean():.0%}")
plt.hist(conf, bins=20, range=(0.5, 1)); plt.xlabel("confidence"); plt.title("model confidence on ambiguous halves"); plt.show()

## 9 · Export to ONNX + a ready-to-use predictor

ONNX + `onnxruntime` is a few MB, so the Hugging Face Space does not need PyTorch. The temperature is baked in, so the output is already calibrated probabilities.

In [ ]:
import json, shutil
class Calibrated(nn.Module):
    def __init__(self, m, T): super().__init__(); self.m, self.T = m, T
    def forward(self, x): return torch.softmax(self.m(x) / self.T, dim=1)
export = Calibrated(model.eval().cpu(), T)
torch.onnx.export(export, torch.zeros(1, 3, 128, 128), "ml2_out/revolve_or_mirror.onnx",
                  input_names=["image"], output_names=["probs"], dynamic_axes={"image": {0: "batch"}, "probs": {0: "batch"}},
                  opset_version=17, dynamo=False)
shutil.copy("ml2_halves.py", "ml2_out/ml2_halves.py")
meta = dict(classes=NAMES, temperature=T, input="3x128x128 render from ml2_halves.render, ImageNet-normalised",
            mean=MEAN.flatten().tolist(), std=STD.flatten().tolist(), epochs=EPOCHS,
            n_train=len(ds["train"]), results=results.to_dict("records"), by_group=by.to_dict("records"))
json.dump(meta, open("ml2_out/meta.json", "w"), indent=2)
model.to(DEVICE)
print(os.listdir("ml2_out"))

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession("ml2_out/revolve_or_mirror.onnx")
def predict_kind(half_points_mm, axis_point, axis_dir):
    """What the app calls: the drawn half + its centreline -> ('revolve' | 'mirror', confidence)."""
    img = H.render(H.shape_from_drawing(half_points_mm, axis_point, axis_dir))
    x = ((img.astype(np.float32) / 255.0 - MEAN.numpy()) / STD.numpy())[None].astype(np.float32)
    p = sess.run(None, {"image": x})[0][0]
    return NAMES[int(p.argmax())], float(p.max())

# 1) two things people draw when they mean "turned"
a = np.linspace(-np.pi / 2, np.pi / 2, 40)
ball = np.stack([30 * np.cos(a), 30 * np.sin(a)], 1)                                   # half a ball
vase = np.stack([12 + 8 * np.sin(np.linspace(0, 3, 30)), np.linspace(0, 70, 30)], 1)    # half a vase
for name, pts in [("half ball", ball), ("half vase", vase)]:
    print(f"{name:12s} -> {predict_kind(pts, [0, 0], [0, 1])}")

# 2) real mirror parts from the test split, fed in the way the APP feeds them (a drawn polyline + axis)
mir = [it for it in items["test"] if it["group"] == "mirror (cad)"][:40]
got = []
for it in mir:
    ring = it["shape"][0][0]; pts = ring[ring[:, 0] > 1e-3 * ring[:, 0].max()]
    got.append(predict_kind(pts * 60 / max(np.ptp(ring[:, 1]), 1e-9), [0, 0], [0, 1])[0])
print(f"mirror test parts via the app's drawing path: {got.count('mirror')}/{len(got)} called mirror")

# 3) honest caveat: a simple straight-edged half (a T, a trapezoid) really can be either a
#    flanged shaft or a flat bracket. In the CAD data such shapes are mostly turned parts, so the
#    model leans "revolve" on them - which is why the app shows both previews and the person picks.
for name, pts in [("half T", np.array([[0, 0], [40, 0], [40, 8], [12, 8], [12, 50], [0, 50]], float)),
                  ("half trapezoid", np.array([[0, 0], [40, 0], [25, 30], [0, 30]], float))]:
    print(f"{name:15s} -> {predict_kind(pts, [0, 0], [0, 1])}   (genuinely ambiguous)")

## 10 · Push the model to the Hugging Face Hub

Uploads the ONNX model, `ml2_halves.py` (the render the app must use), `meta.json`, the results figure and a model card. Set `PUSH_TO_HUB = True`, run, and paste a Hugging Face **write** token when asked.

In [ ]:
res_md = results.to_markdown(index=False); by_md = by.to_markdown(index=False)
card = f"""---
license: other
license_name: fusion-360-gallery-dataset-license
license_link: https://github.com/AutodeskAILab/Fusion360GalleryDataset
library_name: onnx
tags: [sketch, cad, revolve, image-classification, resnet18, fine-tuned]
datasets: [{DATASET}]
---
# sketch2stl · revolve or mirror-extrude? (ML2)

In the sketch2stl app's *half + centreline* mode a person draws one half of an outline. This model suggests
whether that half should be **revolved** around the centreline or **mirrored and extruded**, with a calibrated
confidence. The person always picks; the app shows both 3-D previews when the model is unsure.

**Fine-tuned**: torchvision ResNet-18 (ImageNet weights), all layers, {EPOCHS} epochs, AdamW + one-cycle,
class-balanced sampling, temperature scaling (T = {T:.3f}) fitted on the validation split.

## Data
* revolve: Fusion 360 Gallery designs that are solids of revolution, half = exact mesh cross-section
  (checked: Pappus volume within 5 % of mesh volume), plus procedurally generated turned parts
* mirror: mirror-symmetric extrude profiles (IoU ≥ 0.995) cut along the axis
* ambiguous halves (a rectangle against the axis) excluded from training and scoring
* split by CAD project; hand-wobble added to every picture

## Results (held-out test split)
{res_md}

By group:
{by_md}

## Use
```python
import onnxruntime as ort, ml2_halves as H   # ml2_halves.py is in this repo
img = H.render(H.shape_from_drawing(half_points_mm, axis_point, axis_dir))   # 3x128x128 uint8
x = ((img / 255.0 - mean) / std)[None].astype("float32")                     # mean/std in meta.json
probs = ort.InferenceSession("revolve_or_mirror.onnx").run(None, {{"image": x}})[0][0]   # [mirror, revolve]
```

## Limitations
Revolve examples from CAD are all straight steps (they were built from extrudes); curved turned parts come
from a generator. No real hand-drawn halves were collected, so performance on real drawings is not measured.

## Licence
Derived from the Fusion 360 Gallery Dataset (Autodesk) — non-commercial research use only.
"""
open("ml2_out/README.md", "w").write(card); print(card[:2500])

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, create_repo, upload_folder
    notebook_login()
    create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
    upload_folder(repo_id=MODEL_REPO, folder_path="ml2_out", repo_type="model",
                  commit_message="ML2 revolve-or-mirror ResNet-18 (ONNX) + model card")
    print("pushed -> https://huggingface.co/" + MODEL_REPO)
else:
    print("PUSH_TO_HUB is False - nothing uploaded.")